In [ ]:
import os
import random
import shutil
import requests
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser, NeighborSearch
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB import PDBIO, Select
from tqdm.auto import tqdm

In [ ]:
# ========================== 1. Filtering Criteria  =============================
MAX_RESOLUTION = 3.5
MIN_PEPTIDE_LEN = 3
MAX_PEPTIDE_LEN = 30
REC_PEP_LEN_RATIO = 2.0
MAX_CHAIN_BREAK_DIST = 4.0
MIN_INTERACTION_DIST = 5.0
MAX_RECEPTOR_LEN = 6000

# ========================== 2. Path configuration =============================
PEPBDB_LIST_FILE = "/data/PepBDB/peptidelist.txt"
PEPBDB_PDB_DIR = "/data/PepBDB/pepbdb"
QBIOLIP_LIG_DIR = "/data/Q_BioLip/nonredund_lig"
QBIOLIP_REC_DIR = "/data/Q_BioLip/nonredund_rec"
QBIOLIP_INFO_FILE = "/data/Q_BioLip/PIII_nonredund.txt"
QBIOLIP_ANNOTATION_FILE = "/data/Q_BioLip/Nonredund_annotations.csv"
PROPEDIA_PEP_DIR = "/data/Propedia_v2.3/peptide"
PROPEDIA_REC_DIR = "/data/Propedia_v2.3/receptor"
PROPEDIA_COMPLEX_DIR = "/data/Propedia_v2.3/complex"

# --- Output Path ---
PEPBDB_OUTPUT_DIR = Path("/home/qiuyk/data/PPQ/PPQ_PepBDB")
QBIOLIP_OUTPUT_DIR = Path("/home/qiuyk/data/PPQ/PPQ_Q_BioLip")
PROPEDIA_OUTPUT_DIR = Path("/home/qiuyk/data/PPQ/PPQ_Propedia")
# The final merged main folder
FINAL_DATASET_DIR = Path("/home/qiuyk/data/PPQ/PPQ_dataset")
# Final generated metadata file name
METADATA_CSV_FILE = FINAL_DATASET_DIR.parent / "PPQ_dataset_metadata.csv"

# ========================== 3. random seed =============================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
print(f"The random number seed has been set to: {RANDOM_SEED}")

In [ ]:
# --- Auxiliary Functions for PDB Processing ---
PDB_PARSER = PDBParser(QUIET=True)

def get_structure_length(structure):
    """Calculate the total number of 20 standard amino acid residues in the structure."""
    return sum(1 for res in structure.get_residues() if is_aa(res, standard=True))

def contains_only_standard_amino_acids(structure):
    """
    Check whether all the non-water residues in a structure are among the 20 standard amino acids.
    Check all the residues, not just the ATOM records,
    thus ensuring a strict purity screening for all data sets. 
    """
    for residue in structure.get_residues():
       # Obtain the residue name and remove the spaces at both ends, for example, ' HOH' -> 'HOH'
        res_name = residue.get_resname().strip()
        
        # Allow the presence of water molecules (HOH, WAT), as they will be removed by the function clean_and_save_pdb subsequently.
        if res_name in ['HOH', 'WAT']:
            continue
            
        # For all non-water residues, if it is not a standard amino acid, the structure is considered to be impure.
        if not is_aa(residue, standard=True):
            return False
            
    return True
    
def get_chains_and_length(pdb_path: Path):
    "Extract all chain IDs and the total length from the PDB file." ""
    structure = PDB_PARSER.get_structure(pdb_path.stem, str(pdb_path))
    chain_ids = sorted([chain.id for chain in structure.get_chains()])
    length = get_structure_length(structure)
    return chain_ids, length

def check_structure_integrity(structure, max_dist=4.0):
    """
    Check structural integrity: 
    1. Ensure that there is at least one standard amino acid in the structure. 
    2. Only include 20 standard amino acids. 
    3. There are no broken chains. 
    """
    found_at_least_one_std_residue = False
    for model in structure:
        for chain in model:
            residues = [res for res in chain if is_aa(res, standard=True)]
            if not residues:
                continue

            # If we can reach this point, it indicates that at least one standard amino acid has been identified.
            found_at_least_one_std_residue = True

            # Check for broken links
            for i in range(len(residues) - 1):
                try:
                    dist = residues[i]['CA'] - residues[i+1]['CA']
                    if dist > max_dist: return False
                except KeyError:
                    return False
    return found_at_least_one_std_residue

def get_pdb_resolution_online(pdb_id, cache):
    """Query the PDB resolution online and use the provided cache dictionary."""
    pdb_id = pdb_id.lower()
    if pdb_id in cache: return cache[pdb_id]

    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            res_list = data.get("rcsb_entry_info", {}).get("resolution_combined")
            if res_list and res_list[0] is not None:
                resolution = float(res_list[0])
                cache[pdb_id] = resolution
                return resolution
    except requests.exceptions.RequestException: pass
    cache[pdb_id] = None
    return None

# def convert_hetatm_to_atom(input_path, output_path):
#     """Read the PDB file, convert the HETATM records into ATOM records and add TER."""
#     with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
#         for line in infile:
#             if line.startswith("HETATM"): outfile.write("ATOM  " + line[6:])
#             else: outfile.write(line)
#         outfile.write("TER\n")

class StandardAminoAcidSelector(Select):
    """
    The selector class is used to retain only the standard amino acid residues when saving the PDB.
    It skips all HETATM records (such as water, ligands, ions, etc.) and non-standard amino acids. 
    """
    def accept_residue(self, residue):
        # is_aa(residue, standard=True) checks whether the residue is one of the 20 standard amino acids.
        # At the same time, we use the condition residue.id[0] == ' ' to ensure that we only accept residues that are not HETATM.
        # (The first element of the ID tuple of the HETATM residue is usually 'H_' + LIGAND_NAME)
        if is_aa(residue, standard=True):
            return 1
        else:
            return 0
            
def clean_and_save_pdb(input_path, output_path):
    """
    Read the PDB file, remove all non-standard amino acids and HETATM, and then save the clean structure.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(input_path).stem, input_path)
    
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(output_path), select=StandardAminoAcidSelector())

In [ ]:
def process_pepbdb():
    print("--- Step 1: Start processing the PepBDB dataset ---")
    os.makedirs(PEPBDB_OUTPUT_DIR, exist_ok=True)

    try:
        df = pd.read_csv(PEPBDB_LIST_FILE, sep='\s+', header=None, on_bad_lines='skip',
                         names=['pdb_chain', 'pep_chain', 'pep_len_raw', 'pep_atoms',
                                'rec_chain1', 'rec_chain2', 'rec_atoms', 'x1', 'x2',
                                'resolution', 'type'],
                         usecols=range(11))
        df['pdb_id'] = df['pdb_chain'].str[:4]
        df_filtered = df[(df['resolution'] <= MAX_RESOLUTION) &
                    # The length here includes the non-standard amino acids. Further investigation of the peptide length is still needed!
                         (df['pep_len_raw'] >= MIN_PEPTIDE_LEN) &
                         (df['pep_len_raw'] <= MAX_PEPTIDE_LEN) &  
                         (df['type'] == 'prot')].copy()
    except Exception as e:
        print(f"Error: Failed to read or initially process {PEPBDB_LIST_FILE}: {e}")
        return

    print(f"After the initial screening, there are {len(df_filtered)} candidate complexes.")

    successful_complex_count = 0
    for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="Processing PepBDB"):
        sample_id = f"{row['pdb_id']}_{row['pep_chain']}"
        receptor_path = os.path.join(PEPBDB_PDB_DIR, sample_id, 'receptor.pdb')
        peptide_path = os.path.join(PEPBDB_PDB_DIR, sample_id, 'peptide.pdb')

        if not (os.path.exists(receptor_path) and os.path.exists(peptide_path)): continue

        try:
            receptor_struct = PDB_PARSER.get_structure('R', receptor_path)
            peptide_struct = PDB_PARSER.get_structure('P', peptide_path)

            if not (contains_only_standard_amino_acids(receptor_struct) and contains_only_standard_amino_acids(peptide_struct)): continue

            receptor_len = get_structure_length(receptor_struct)
            if receptor_len > MAX_RECEPTOR_LEN: continue

            peptide_len = get_structure_length(peptide_struct)
            if receptor_len < peptide_len * REC_PEP_LEN_RATIO: continue

            if not (MIN_PEPTIDE_LEN <= peptide_len <= MAX_PEPTIDE_LEN): continue

            if not (check_structure_integrity(receptor_struct) and check_structure_integrity(peptide_struct)): continue
            receptor_atoms = list(receptor_struct.get_atoms())
            peptide_atoms = list(peptide_struct.get_atoms())
            if not receptor_atoms or not peptide_atoms: continue

            ns = NeighborSearch(receptor_atoms)
            if not any(ns.search(atom.get_coord(), MIN_INTERACTION_DIST, level='A') for atom in peptide_atoms):
                continue

            target_dir = PEPBDB_OUTPUT_DIR / sample_id
            os.makedirs(target_dir, exist_ok=True)
            clean_and_save_pdb(receptor_path, target_dir / 'receptor.pdb')
            clean_and_save_pdb(peptide_path, target_dir / 'peptide.pdb')
            successful_complex_count += 1
        except Exception:
            continue

    print(f"\n--- The PepBDB dataset processing is complete ---")
    print(f"A total of {successful_complex_count} complex samples were selected and saved in: {PEPBDB_OUTPUT_DIR}")

# ===== Carry out the PepBDB processing =====
process_pepbdb()

In [ ]:
def process_qbiolip():
    print("\n--- Step 2: Start processing the Q_BioLip dataset ---")
    os.makedirs(QBIOLIP_OUTPUT_DIR, exist_ok=True)

    pepbdb_complexes = os.listdir(PEPBDB_OUTPUT_DIR)
    pepbdb_ids = {c.split('_')[0] for c in pepbdb_complexes}
    print(f"Excluding {len(pepbdb_ids)} PDB IDs that have already been processed in PepBDB.")

    try:
        df_annot = pd.read_csv(QBIOLIP_ANNOTATION_FILE, on_bad_lines='skip')
        df_annot.drop_duplicates(subset=['Assembly ID'], inplace=True)
        annot_map = df_annot.set_index('Assembly ID')[['Resolution (Å)', 'PDB ID']].to_dict('index')

        df_pep_info = pd.read_csv(QBIOLIP_INFO_FILE)
        candidates = df_pep_info[df_pep_info['Assembly_ID'].isin(annot_map.keys())].copy()
        candidates['pdb_id'] = candidates['Assembly_ID'].str.split('_').str[0]
        candidates['resolution'] = candidates['Assembly_ID'].map(lambda x: annot_map.get(x, {}).get('Resolution (Å)'))

        candidates.dropna(subset=['resolution'], inplace=True)
        candidates = candidates[(candidates['pdb_id'].str.lower().isin(pepbdb_ids) == False) &
                                (candidates['resolution'] <= MAX_RESOLUTION)]

    except Exception as e:
        print(f"Error: Failed to read or preliminarily process Q_BioLip metadata: {e}")
        return

    print(f"After the initial screening, there are {len(candidates)} candidate complexes.")

    successful_complex_count = 0
    for _, row in tqdm(candidates.iterrows(), total=len(candidates), desc="Processing Q_BioLip"):
        assembly_id, ligand_file = row['Assembly_ID'], row['Ligand_file']
        receptor_path = os.path.join(QBIOLIP_REC_DIR, f"{assembly_id}.pdb")
        peptide_path = os.path.join(QBIOLIP_LIG_DIR, f"{ligand_file}.pdb")

        if not (os.path.exists(receptor_path) and os.path.exists(peptide_path)): continue

        try:
            receptor_struct = PDB_PARSER.get_structure('R', receptor_path)
            peptide_struct = PDB_PARSER.get_structure('P', peptide_path)

            if not (contains_only_standard_amino_acids(receptor_struct) and contains_only_standard_amino_acids(peptide_struct)): continue

            peptide_len = get_structure_length(peptide_struct)
            if not (MIN_PEPTIDE_LEN <= peptide_len <= MAX_PEPTIDE_LEN): continue

            receptor_len = get_structure_length(receptor_struct)
            if receptor_len > MAX_RECEPTOR_LEN: continue
            if receptor_len < peptide_len * REC_PEP_LEN_RATIO: continue
            if not (check_structure_integrity(receptor_struct) and check_structure_integrity(peptide_struct)): continue

            receptor_atoms = list(receptor_struct.get_atoms())
            peptide_atoms = list(peptide_struct.get_atoms())
            if not receptor_atoms or not peptide_atoms: continue
            ns = NeighborSearch(receptor_atoms)
            if not any(ns.search(atom.get_coord(), MIN_INTERACTION_DIST, level='A') for atom in peptide_atoms):
                continue

            peptide_chain_id = ligand_file.split('_')[-1]
            output_sample_name = f"{assembly_id}_{peptide_chain_id}"
            target_dir = QBIOLIP_OUTPUT_DIR / output_sample_name
            os.makedirs(target_dir, exist_ok=True)

            clean_and_save_pdb(receptor_path, target_dir / 'receptor.pdb')
            clean_and_save_pdb(peptide_path, target_dir / 'peptide.pdb')
            successful_complex_count += 1
        except Exception:
            continue

    print(f"\n--- The Q_BioLip dataset has been processed. ---")
    print(f"A total of {successful_complex_count} complex samples were selected and saved in: {QBIOLIP_OUTPUT_DIR}")

# ===== Perform Q_BioLip processing =====
process_qbiolip()

In [ ]:
def process_propedia():
    print("\n--- Step 3: Start processing the Propedia_v2.3 dataset ---")
    os.makedirs(PROPEDIA_OUTPUT_DIR, exist_ok=True)

    # Obtain the processed PDB IDs
    pepbdb_complexes = os.listdir(PEPBDB_OUTPUT_DIR)
    qbiolip_complexes = os.listdir(QBIOLIP_OUTPUT_DIR)
    processed_ids = {c.split('_')[0] for c in pepbdb_complexes} | \
                    {c.split('_')[0] for c in qbiolip_complexes}
    print(f"Excluding {len(processed_ids)} PDB IDs that have been processed in the previous dataset.")

    try:
        complex_files = [f for f in os.listdir(PROPEDIA_COMPLEX_DIR) if f.endswith('.pdb')]
        # Pre-scanning to establish a case-insensitive file index
        peptide_file_map = {f.lower(): f for f in os.listdir(PROPEDIA_PEP_DIR)}
        receptor_file_map = {f.lower(): f for f in os.listdir(PROPEDIA_REC_DIR)}
    except FileNotFoundError as e:
        print(f"Error: Unable to locate the Propedia_v2.3 data folder: {e.filename}")
        return

    print(f"From Propedia_v2.3, {len(complex_files)} initial candidate complexes were found.")

    resolution_cache = {}
    successful_complex_count = 0
    for complex_filename in tqdm(complex_files, desc="Processing Propedia_v2.3"):
        try:
            parts = complex_filename.replace('.pdb', '').split('_')
            if len(parts) != 3: continue
            pdb_id, peptide_chain, receptor_chain = parts[0].lower(), parts[1], parts[2]
        except Exception: continue

        if pdb_id in processed_ids: continue

        resolution = get_pdb_resolution_online(pdb_id, resolution_cache)
        if resolution is None or resolution > MAX_RESOLUTION: continue

        peptide_key = f"{pdb_id}_{peptide_chain}.pdb".lower()
        receptor_key = f"{pdb_id}_{receptor_chain}.pdb".lower()

        original_peptide_file = peptide_file_map.get(peptide_key)
        original_receptor_file = receptor_file_map.get(receptor_key)

        if not (original_peptide_file and original_receptor_file): continue

        receptor_path = os.path.join(PROPEDIA_REC_DIR, original_receptor_file)
        peptide_path = os.path.join(PROPEDIA_PEP_DIR, original_peptide_file)

        try:
            receptor_struct = PDB_PARSER.get_structure('R', receptor_path)
            peptide_struct = PDB_PARSER.get_structure('P', peptide_path)

            if not (contains_only_standard_amino_acids(receptor_struct) and contains_only_standard_amino_acids(peptide_struct)): continue
                
            peptide_len = get_structure_length(peptide_struct)
            if not (MIN_PEPTIDE_LEN <= peptide_len <= MAX_PEPTIDE_LEN): continue

            receptor_len = get_structure_length(receptor_struct)
            if receptor_len > MAX_RECEPTOR_LEN: continue
            if receptor_len < peptide_len * REC_PEP_LEN_RATIO: continue
            if not (check_structure_integrity(receptor_struct) and check_structure_integrity(peptide_struct)): continue

            receptor_atoms = list(receptor_struct.get_atoms())
            peptide_atoms = list(peptide_struct.get_atoms())
            if not receptor_atoms or not peptide_atoms: continue
            ns = NeighborSearch(receptor_atoms)
            if not any(ns.search(atom.get_coord(), MIN_INTERACTION_DIST, level='A') for atom in peptide_atoms): continue

            output_sample_name = complex_filename.replace('.pdb', '')
            target_dir = PROPEDIA_OUTPUT_DIR / output_sample_name
            os.makedirs(target_dir, exist_ok=True)
            clean_and_save_pdb(receptor_path, target_dir / 'receptor.pdb')
            clean_and_save_pdb(peptide_path, target_dir / 'peptide.pdb')
            successful_complex_count += 1
        except Exception:
            continue

    print(f"\n--- Propedia_v2.3 dataset processing completed ---")
    print(f"A total of {successful_complex_count} complex samples were selected and saved in: {PROPEDIA_OUTPUT_DIR}")

    return resolution_cache

# ===== Execute Propedia_v2.3 processing =====
propedia_resolution_cache = process_propedia()

In [ ]:
import tempfile

def rename_pdb_chains(pdb_path: Path, old_chain_id: str, new_chain_id: str) -> bool:
    """
    Modify the specified old chain ID to the new chain ID in the PDB file. Directly modify the file content.
    """
    if not pdb_path.exists():
        return False
    
    with open(pdb_path, 'r') as f:
        lines = f.readlines()
        
    updated_lines = []
    chains_modified = False
    
    for line in lines:
        # The chain ID of the PDB file is in the 22nd column (index 21)
        if line.startswith(("ATOM", "HETATM")) and len(line) > 21:
            current_chain = line[21]
            if current_chain == old_chain_id:
                # Replace Chain ID
                line = line[:21] + new_chain_id[0] + line[22:]
                chains_modified = True
        updated_lines.append(line)

    if chains_modified:
        # Rewrite the file
        with open(pdb_path, 'w') as f:
            f.writelines(updated_lines)
    
    return chains_modified

def get_available_chain_id(used_chains: set) -> str:
    """
    Return the first single-character PDB format chain ID that does not appear in the used_chains set.
    Order of attempts: digits (0-9), lowercase letters (a-z), and then uppercase letters (Z, Y, X...) 。
    """
    
    # 1. Try numbers (0-9)
    for i in range(10):
        chain_id = str(i)
        if chain_id not in used_chains:
            return chain_id

    # 2. Try lowercase letters (a-z)
    for char_code in range(ord('a'), ord('z') + 1):
        chain_id = chr(char_code)
        if chain_id not in used_chains:
            return chain_id
            
    # 3. Try using capital letters (including less common ones such as Z, Y, X... as supplementary options)
    for char_code in range(ord('Z'), ord('A') - 1, -1):
        chain_id = chr(char_code)
        if chain_id not in used_chains:
            return chain_id
            
    # Final Insurance ID (if all the letters and numbers have been used up in the chain ID of the PDB file)
    return '!'
    
def merge_rename_and_generate_metadata(propedia_resolution_cache):
    print("\n--- Step 4: Start merging, renaming and generating metadata ---")

    source_dirs_map = {
        "PepBDB": PEPBDB_OUTPUT_DIR,
        "Q_BioLip": QBIOLIP_OUTPUT_DIR,
        "Propedia": PROPEDIA_OUTPUT_DIR
    }

    # --- Step 4.1: Preload information ---
    print("Step 4.1: Preload resolution information...")

    resolution_cache = {}
    df_existing_meta = None
    if METADATA_CSV_FILE.exists():
        try:
            df_existing_meta = pd.read_csv(METADATA_CSV_FILE)
            # Load the PDB ID and resolution into the cache dictionary
            for _, row in df_existing_meta.iterrows():
                if pd.notna(row['pdb_id']) and pd.notna(row['resolution']):
                    resolution_cache[str(row['pdb_id']).lower()] = row['resolution']
            print(f"  - Success! {len(resolution_cache)} cached resolutions were loaded from {METADATA_CSV_FILE}.")
        except Exception as e:
            print(f"  - Warning: Failed to read the existing metadata file: {e}")
    else:
        print("  - No existing metadata file was found. A new one will be created.")

    pepbdb_res_map, qbiolip_res_map = {}, {}
    try:
        with open(PEPBDB_LIST_FILE, 'r') as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 8: pepbdb_res_map[f"{parts[0]}_{parts[1]}"] = float(parts[-2])
    except FileNotFoundError: print("  - Warning: Unable to locate the peptidelist.txt file of PepBDB.")
    try:
        df_annot = pd.read_csv(QBIOLIP_ANNOTATION_FILE, on_bad_lines='skip')
        df_annot.drop_duplicates(subset=['Assembly ID'], inplace=True)
        for _, row in df_annot.iterrows(): qbiolip_res_map[row['Assembly ID']] = row['Resolution (Å)']
    except (FileNotFoundError, KeyError): print("  - Warning: Unable to locate or parse the annotation file for Q_BioLip.")

    # --- Step 4.2: Main Process ---
    print(f"Step 4.2: Begin processing, moving, and renaming the complex...")
    os.makedirs(FINAL_DATASET_DIR, exist_ok=True)
    all_metadata = []

    for source_name, source_path in source_dirs_map.items():
        print(f"--- Processing source: {source_name} ---")
        if not source_path.is_dir(): continue

        for item_name in tqdm(os.listdir(source_path), desc=f"Processing {source_name}"):
            source_item_path = source_path / item_name
            if not source_item_path.is_dir(): continue

            temp_dir = None # Temporary directory used for renaming Q_BioLip
            
            try:
                receptor_path_src = source_item_path / 'receptor.pdb'
                peptide_path_src = source_item_path / 'peptide.pdb'
                
                if not (receptor_path_src.exists() and peptide_path_src.exists()): continue

                current_item_path = source_item_path
                rec_chains, rec_len = get_chains_and_length(receptor_path_src)
                pep_chains, pep_len = get_chains_and_length(peptide_path_src)
                rec_chains_set = set(rec_chains)
                pep_chains_set = set(pep_chains)
                
                is_qbiolip_conflict = (source_name == "Q_BioLip") and not rec_chains_set.isdisjoint(pep_chains_set)

                if is_qbiolip_conflict:
                    
                    # Assume that the Q_BioLip peptide has only one chain (this is the most common situation)
                    if len(pep_chains) == 1:
                        old_pep_chain_id = pep_chains[0]
                        
                        # Find the first available new chain ID
                        new_pep_chain_id = get_available_chain_id(rec_chains_set)
                        
                        # Copy to Temporary Directory
                        temp_dir = Path(tempfile.mkdtemp())
                        shutil.copytree(str(source_item_path), str(temp_dir), dirs_exist_ok=True) 
                        current_item_path = temp_dir
                        
                        # Rename the peptide chain IDs in the temporary directory
                        rename_pdb_chains(current_item_path / 'peptide.pdb', old_pep_chain_id, new_pep_chain_id)
                        
                        # Update the chain information for subsequent checks and naming
                        pep_chains = [new_pep_chain_id]
                        pep_chains_set = set(pep_chains)
                    else:
                        # Q_BioLip peptide has multiple chains and there are conflicts. Maintain the original logic and skip.
                        continue

                # Recheck the chain ID conflicts (for PepBDB, Propedia)
                if not rec_chains_set.isdisjoint(pep_chains_set):
                    if temp_dir: shutil.rmtree(temp_dir)
                    continue

                pdb_id = item_name.split('_')[0].lower()

                resolution = None
                if source_name == "PepBDB": resolution = pepbdb_res_map.get(item_name)
                elif source_name == "Q_BioLip":
                    assembly_id = '_'.join(item_name.split('_')[:2])
                    resolution = qbiolip_res_map.get(assembly_id)
                elif source_name == "Propedia":
                    resolution = propedia_resolution_cache.get(pdb_id)

                # Create a new complex ID using the chain ID
                new_name = f"{pdb_id}_{''.join(rec_chains)}_{''.join(pep_chains)}"
                destination_path = FINAL_DATASET_DIR / new_name
                
                if destination_path.exists():
                    if temp_dir: shutil.rmtree(temp_dir) 
                    continue

                # Copy to final directory
                shutil.copytree(str(current_item_path), str(destination_path))
                
                # Clean up the temporary directory
                if temp_dir: shutil.rmtree(temp_dir)

                all_metadata.append({
                    "complex_id": new_name, "pdb_id": pdb_id,
                    "receptor_chains": ','.join(rec_chains), "peptide_chains": ','.join(pep_chains),
                    "receptor_length": rec_len, "peptide_length": pep_len,
                    "resolution": resolution, "source_dataset": source_name, "original_id": item_name
                })
                
            except Exception as e: 
                print(f"Error: An error occurred while processing '{item_name}': {e}")
                if temp_dir and os.path.exists(temp_dir): shutil.rmtree(temp_dir) # Ensure that errors are also cleaned up

    print(f"\nStep 4.3: The final metadata file is being saved...")
    if all_metadata:
        df_new_metadata = pd.DataFrame(all_metadata)

        if df_existing_meta is not None:
            df_final = pd.concat([df_existing_meta, df_new_metadata]).drop_duplicates(subset=['complex_id'], keep='last')
        else:
            df_final = df_new_metadata

        df_final.to_csv(METADATA_CSV_FILE, index=False)
        print(f"The final metadata has been successfully saved to: {METADATA_CSV_FILE}")
    else:
        print("There were no new or updated metadata during this run.")

    print("\n--- Merge and metadata generation completed! ---")
    print(f"All complexes have been merged into: {FINAL_DATASET_DIR}")

if propedia_resolution_cache is not None:
    merge_rename_and_generate_metadata(propedia_resolution_cache)
else:
    print("\nPropedia processing failed. The merge step has been skipped.")

In [ ]:
import os
import pickle
import glob
import numpy as np
import prody
from tqdm.auto import tqdm

RESTYPE_3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V",
}
STANDARD_AMINO_ACIDS_3 = set(RESTYPE_3_TO_1.keys())
ATOM_ORDER = {
    "N": 0, "CA": 1, "C": 2, "CB": 3, "O": 4, "CG": 5, "CG1": 6, "CG2": 7,
    "OG": 8, "OG1": 9, "SG": 10, "CD": 11, "CD1": 12, "CD2": 13, "ND1": 14,
    "ND2": 15, "OD1": 16, "OD2": 17, "SD": 18, "CE": 19, "CE1": 20, "CE2": 21,
    "CE3": 22, "NE": 23, "NE1": 24, "NE2": 25, "OE1": 26, "OE2": 27, "CH2": 28,
    "NH1": 29, "NH2": 30, "OH": 31, "CZ": 32, "CZ2": 33, "CZ3": 34, "NZ": 35,
    "OXT": 36,
}
MAX_ATOMS = len(ATOM_ORDER)

def extract_chain_features(chain_selection: prody.Selection, chain_idx: int, fallback_chain_id: str = None):
    residues = [res for res in chain_selection.getHierView().iterResidues() if
                res.getResname() in STANDARD_AMINO_ACIDS_3]
    num_residues = len(residues)

    chain_ids = list(set(chain_selection.getChids()))
    chain_id = chain_ids[0] if chain_ids else fallback_chain_id

    xyz_37 = np.zeros((num_residues, MAX_ATOMS, 3), dtype=np.float32)
    mask_37 = np.zeros((num_residues, MAX_ATOMS), dtype=bool)
    res_indices = np.zeros(num_residues, dtype=np.int32)
    sequence_list = []

    for i, res in enumerate(residues):
        res_indices[i] = res.getResnum()
        sequence_list.append(RESTYPE_3_TO_1.get(res.getResname(), 'X'))
        for atom in res:
            atom_name = atom.getName()
            if atom_name in ATOM_ORDER:
                atom_idx = ATOM_ORDER[atom_name]
                xyz_37[i, atom_idx] = atom.getCoords()
                mask_37[i, atom_idx] = True
    sequence = "".join(sequence_list)
    backbone_mask = (mask_37[:, ATOM_ORDER['N']] & mask_37[:, ATOM_ORDER['CA']] & mask_37[:, ATOM_ORDER['C']] & mask_37[:, ATOM_ORDER['O']])
    return {
        'xyz_37': xyz_37,
        'xyz_37_mask': mask_37,
        'seq': sequence,
        'R_idx': res_indices,
        'chain_id': chain_id,
        'chain_idx': chain_idx,
        'mask': backbone_mask
    }

def process_folder(folder_path: str, pkl_dir: str) -> bool:
    folder_name = os.path.basename(folder_path)
    parts = folder_name.split('_')
    if len(parts) != 3:
        tqdm.write(f"Skip {folder_path}: The folder name format is incorrect. It should be pdbID_receptorChain_peptideChain.")
        return False
    pdb_id, receptor_chain_id, peptide_chain_id = parts

    receptor_pdb_path = os.path.join(folder_path, 'receptor.pdb')
    peptide_pdb_path = os.path.join(folder_path, 'peptide.pdb')
    if not (os.path.exists(receptor_pdb_path) and os.path.exists(peptide_pdb_path)):
        tqdm.write(f"Skip {folder_path}: Missing receptor.pdb or peptide.pdb")
        return False

    try:
        receptor_struct = prody.parsePDB(receptor_pdb_path)
        peptide_struct = prody.parsePDB(peptide_pdb_path)
    except Exception as e:
        tqdm.write(f"Skipping {folder_path}: PDB file parsing failed, error: {e}")
        return False

    receptor_protein = receptor_struct.select('protein') if receptor_struct else None
    peptide_protein = peptide_struct.select('protein') if peptide_struct else None
    if not (receptor_protein and receptor_protein.numAtoms() > 0 and peptide_protein and peptide_protein.numAtoms() > 0):
        tqdm.write(f"Skip {folder_path}: The receptor or polypeptide lacks effective protein segments")
        return False

    def is_clean(selection: prody.Selection):
        if not selection:
            return False
        try:
            for res in selection.getHierView().iterResidues():
                if res is None or res.getAtom('CA') is None or res.getAtom('N') is None or res.getAtom('C') is None or res.getAtom('O') is None:
                    tqdm.write(f"The chain {list(set(selection.getChids()))[0] if list(set(selection.getChids())) else 'Unknown'} in {folder_path} is not clean and lacks the backbone atoms.")
                    return False
            return True
        except Exception as e:
            tqdm.write(f"The chain {list(set(selection.getChids()))[0] if list(set(selection.getChids())) else 'Unknown'}  failed during the check of {folder_path}: {e}")
            return False

    clean_receptor_chains = []
    try:
        receptor_unique_chids = sorted(list(set(receptor_protein.getChids())))
    except AttributeError:
        receptor_unique_chids = []
        tqdm.write(f"Skip {folder_path}: Unable to obtain the ID of the receptor chain")

    for chid in receptor_unique_chids:
        chain_selection = receptor_protein.select(f'chain {chid}')
        if chain_selection and chain_selection.numAtoms() > 0 and is_clean(chain_selection):
            clean_receptor_chains.append(chain_selection)
        else:
            tqdm.write(f"The receptor chain {chid} has been excluded from {folder_path}: it is either not clean or lacks valid atoms.")

    clean_peptide_chains = []
    try:
        peptide_unique_chids = list(set(peptide_protein.getChids()))
        if len(peptide_unique_chids) != 1:
            tqdm.write(f"Skip {folder_path}: The peptide has multiple chains ({peptide_unique_chids}), but it is expected to have only one chain.")
            return False
        peptide_chid = peptide_unique_chids[0]
        peptide_selection = peptide_protein.select(f'chain {peptide_chid}') if peptide_chid else peptide_protein
        if peptide_selection and peptide_selection.numAtoms() > 0 and is_clean(peptide_selection):
            clean_peptide_chains.append(peptide_selection)
        else:
            tqdm.write(f"The polypeptide chain {peptide_chid if peptide_chid else 'unknown'} was excluded from {folder_path}: it was not clean or did not have valid atoms.")
    except AttributeError:
        tqdm.write(f"Skip {folder_path}: Unable to obtain the peptide chain ID or parsing failed.")
        return False

    if len(clean_receptor_chains) == 0 or len(clean_peptide_chains) == 0:
        tqdm.write(f"Skip {folder_path}: Both a clean receptor chain and a clean polypeptide chain must be present simultaneously（Current receptor: {len(clean_receptor_chains)}, peptide: {len(clean_peptide_chains)}）.")
        return False

    sorted_receptor_chains = sorted(clean_receptor_chains, key=lambda c: list(set(c.getChids()))[0] if list(set(c.getChids())) else receptor_chain_id)
    sorted_peptide_chains = clean_peptide_chains
    sorted_all_chains = sorted_receptor_chains + sorted_peptide_chains

    chain_id_map = {}
    idx = 0
    for ch in sorted_all_chains:
        chids = list(set(ch.getChids()))
        chid = chids[0] if chids else (receptor_chain_id if ch in sorted_receptor_chains else peptide_chain_id)
        chain_id_map[chid] = idx
        idx += 1

    processed_chains_list = []
    for chain_obj in sorted_all_chains:
        current_chids = list(set(chain_obj.getChids()))
        current_chid = current_chids[0] if current_chids else (receptor_chain_id if chain_obj in sorted_receptor_chains else peptide_chain_id)
        fallback_id = receptor_chain_id if chain_obj in sorted_receptor_chains else peptide_chain_id
        chain_features = extract_chain_features(chain_obj, chain_id_map[current_chid], fallback_chain_id=fallback_id)
        processed_chains_list.append(chain_features)

    all_chain_ids = sorted(chain_id_map.keys())
    chain_ids_str = ''.join(all_chain_ids)
    output_pkl_path = os.path.join(pkl_dir, f"{pdb_id}_{chain_ids_str}.pkl")
    if os.path.exists(output_pkl_path):
        tqdm.write(f"Skipping {folder_path}: Pickle file already exists: {output_pkl_path}.")
        return True

    final_data_to_save = {
        'pdb_id': pdb_id,
        'chain_features': processed_chains_list
    }

    with open(output_pkl_path, 'wb') as f:
        pickle.dump(final_data_to_save, f)
    # tqdm.write(f"Successfully processed {folder_path}, and generated pickle file: {output_pkl_path}.")

    return True

def main():
    INPUT_DIR = "/home/qiuyk/data/PPQ/PPQ_dataset"
    OUTPUT_PKL_DIR = "/home/qiuyk/data/PPQ/PPQ_processed_pkls"

    os.makedirs(OUTPUT_PKL_DIR, exist_ok=True)

    subfolders = [f.path for f in os.scandir(INPUT_DIR) if f.is_dir()]
    if not subfolders:
        print(f"Error: No subfolders were found in the directory '{INPUT_DIR}'.")
        return

    print(f"Found {len(subfolders)} subfolders ready for processing...")
    success_count = 0
    for folder in tqdm(subfolders, desc="Processing folders"):
        if process_folder(folder, OUTPUT_PKL_DIR):
            success_count += 1

    print(f"\nAll folders have been processed!")
    print(f"In total, there are {len(subfolders)} subfolders. Successfully processed {success_count} of them, and generated pickle files which are saved in: {OUTPUT_PKL_DIR}.")
    print(f"Number of failures: {len(subfolders) - success_count}")  
    # NOTE：In fact, the actual count is not so high. There are cases of duplicate file names!!!

if __name__ == "__main__":
    prody.confProDy(verbosity='none')
    main()

In [ ]:
PKL_DIR = Path("/home/qiuyk/data/PPQ/PPQ_processed_pkls") 
OUTPUT_DIR = Path("/home/qiuyk/data/PPQ/")

def create_metadata_by_shortest_chain_rule(pkl_dir: Path):
    """
    Traverse the .pkl file and extract the metadata using the rule "the shortest chain is the peptide". 
    The PDB ID is parsed from the file name.
    """
    print(f"--- Step 1: Start extracting metadata from the {pkl_dir} directory according to the 'shortest chain' rule ---")
    
    if not pkl_dir.exists():
        print(f"Error: Directory does not exist {pkl_dir}.")
        return pd.DataFrame()

    pkl_files = list(pkl_dir.glob("*.pkl"))
    if not pkl_files:
        print(f"Error: No .pkl files were found in {pkl_dir}.")
        return pd.DataFrame()

    print(f"Found {len(pkl_files)} .pkl files. Extracting length information...")
    
    metadata_list = []
    for pkl_file in tqdm(pkl_files, desc="Handle .pkl files"):
        try:
            base_name = pkl_file.stem
            pdb_id = base_name.split('_')[0]
            
            with open(pkl_file, 'rb') as f:
                data = pickle.load(f)
            
            chains = data.get('chain_features', [])
            
            if len(chains) < 2:
                tqdm.write(f"Warning: The file {pkl_file.name} contains less than 2 links. Skipping.")
                continue

            chain_lengths = [len(c['seq']) for c in chains]
            
            shortest_chain_idx = np.argmin(chain_lengths)
            
            peptide_length = chain_lengths[shortest_chain_idx]
            
            receptor_length = 0
            for i, length in enumerate(chain_lengths):
                if i != shortest_chain_idx:
                    receptor_length += length
            
            if receptor_length == 0:
                tqdm.write(f"Warning: The receptor length for the file {pkl_file.name} could not be calculated and it has been skipped.")
                continue

            metadata_list.append({
                'pdb_id': pdb_id,
                'receptor_length': receptor_length,
                'peptide_length': peptide_length
            })

        except Exception as e:
            tqdm.write(f"An error occurred while processing the file {pkl_file.name}: {e}.")
            
    return pd.DataFrame(metadata_list)

def plot_distributions(df_meta: pd.DataFrame, output_dir: Path):

    print("\n--- Step 2: Start drawing the length distribution graph based on the extracted metadata ---")
    
    if df_meta.empty:
        print("Error: Metadata is empty, unable to generate graph.")
        return

    output_dir.mkdir(exist_ok=True)

    # --- Figure 1: Distribution of Peptide Length ---
    plt.figure(figsize=(12, 8))
    sns.histplot(data=df_meta, x='peptide_length', kde=True, binwidth=1, color='dodgerblue')
    plt.title('Peptide Length Distribution', fontsize=22, weight='bold')
    plt.xlabel("Peptide Length", fontsize=20, weight='bold')
    plt.ylabel("Count", fontsize=20, weight='bold')
    plt.xlim(3, 30)
    plt.tight_layout()
    output_path_peptide = output_dir / "peptide_length_distribution_from_pkl.svg"
    plt.savefig(output_path_peptide, bbox_inches='tight')
    print(f"The distribution graph of peptide lengths has been saved to: {output_path_peptide}")
    plt.show()

    # --- Figure 2: Distribution of Receptor Length ---
    plt.figure(figsize=(12, 8))
    sns.histplot(data=df_meta, x='receptor_length', kde=True, color='darkorange')
    plt.title('Receptor Length Distribution', fontsize=22, weight='bold')
    plt.xlabel("Receptor Length", fontsize=20, weight='bold')
    plt.ylabel("Count", fontsize=20, weight='bold')
    if not df_meta['receptor_length'].empty:
        upper_limit = df_meta['receptor_length'].quantile(0.995)
        plt.xlim(0, upper_limit)
    plt.tight_layout()
    output_path_receptor = output_dir / "receptor_length_distribution_from_pkl.svg"
    plt.savefig(output_path_receptor, bbox_inches='tight')
    print(f"The receptor length distribution graph has been saved to: {output_path_receptor}")
    plt.show()

    print("\n--- Visualization completed! ---")


def save_pdb_ids_to_file(df: pd.DataFrame, output_path: Path):

    print(f"\n--- Step 3: Save the PDB ID to a file ---")
    if df.empty:
        print("The DataFrame is empty, so the PDB ID cannot be saved.")
        return

    pdb_ids = df['pdb_id'].unique()
    
    with open(output_path, 'w') as f:
        for pdb_id in sorted(pdb_ids):
            f.write(f"{pdb_id}\n")
            
    print(f"Successfully saved a list of {len(pdb_ids)} unique PDB IDs to: {output_path}.")


if __name__ == "__main__":

    df_metadata = create_metadata_by_shortest_chain_rule(PKL_DIR)

    if not df_metadata.empty:
        print("\n--- Metadata extraction preview ---")
        print(df_metadata.head())
        print("\n--- Statistical Summary ---")
        print(df_metadata.describe())
        
        plot_distributions(df_metadata, OUTPUT_DIR)

        pdb_id_list_path = OUTPUT_DIR / "finetune_exclusion_pdb_ids.txt"
        save_pdb_ids_to_file(df_metadata, pdb_id_list_path)